# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiezugwu/Data-Science/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
Finding #1: Logistic regression separates growing and declining pages with 71% holdout accuracy
The reported 71% holdout accuracy is a useful exploratory result. Could you clarify whether the 80/20 split kept clients separate, and whether all input features were measured before the period used to assign the growth label?

Finding #2: Average position is the strongest feature for estimating health score
The paper helpfully discloses that health score is constructed partly from the same inputs used by the model. Could you clarify whether the model’s purpose is to reconstruct that existing score or predict an independent future outcome?




In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# from google.colab import files
# uploaded = files.upload()
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr

D = pd.read_parquet("decline_recovery.parquet")
R = pd.read_parquet("ranking_lifecycle.parquet")[["content_hash_id", "search_volume", "main_intent"]]  # your notebook: read_parquet
D = D.merge(R, on="content_hash_id", how="left")

D["severity"] = -(D.impressions_last_30d / D.impressions_prev_30d - 1) * 100
D["impressions_first30"] = D.impressions_90d - D.impressions_last_30d - D.impressions_prev_30d
D["early_momentum_pct"] = (D.impressions_prev_30d / D.impressions_first30.where(D.impressions_first30 >= 50) - 1) * 100
D["prev30_ctr_pct"] = D.clicks_prev_30d / D.impressions_prev_30d * 100
D["log_prev30_impressions"] = np.log1p(D.impressions_prev_30d)
D["age_at_outcome_start_days"] = D.content_age_days - 30
D["intent"] = D.main_intent.astype("category").cat.codes.replace(-1, np.nan)
D["type"] = D.content_type.astype("category").cat.codes
feats = ["impressions_first30", "early_momentum_pct", "prev30_ctr_pct", "log_prev30_impressions",
         "age_at_outcome_start_days", "search_volume", "intent", "type",
         "keyword_token_count", "public_url_path_depth", "content_title_token_count"]
K = 500

def precision_at_k(order_index, is_severe, k=K):
    return is_severe.loc[order_index[:k]].mean()

def run(seed, grouped):
    if grouped:   # AFTER: whole clients are held out
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                      .split(D, groups=D.client_hash_id))
    else:         # BEFORE: random rows, so the same client is in train and test
        tr, te = next(ShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(D))
    train, test = D.iloc[tr], D.iloc[te]
    severe = test.severity >= train.severity.quantile(0.75)      # cutoff from train only

    model = RandomForestRegressor(n_estimators=200, min_samples_leaf=50, n_jobs=-1, random_state=42)
    model.fit(train[feats], train.severity)

    b = test.assign(new_page=(test.age_at_outcome_start_days < 60).astype(int),
                    no_history=(test.impressions_first30 < 50).astype(int))
    b["score"] = b.new_page + b.no_history
    base_order = b.sort_values(["score", "impressions_first30", "impressions_prev_30d"],
                               ascending=[False, True, False]).index
    pred = pd.Series(model.predict(test[feats]), index=test.index)

    return {"split": "grouped by client" if grouped else "random rows", "seed": seed,
            "test_rows": len(test), "test_clients": test.client_hash_id.nunique(),
            "base_rate": severe.mean(),
            "baseline": precision_at_k(base_order, severe),
            "model": precision_at_k(pred.sort_values(ascending=False).index, severe),
            "spearman": spearmanr(pred, test.severity)[0]}

rows = pd.DataFrame([run(s, g) for g in (False, True) for s in (1, 2, 3, 4, 42)]).round(3)
print(rows.to_string(index=False))
print(rows.groupby("split")[["base_rate", "baseline", "model", "spearman"]].mean().round(3))

            split  seed  test_rows  test_clients  base_rate  baseline  model  spearman
      random rows     1      11251            42      0.248     0.328  0.814     0.454
      random rows     2      11251            41      0.253     0.332  0.838     0.471
      random rows     3      11251            42      0.252     0.340  0.820     0.465
      random rows     4      11251            41      0.253     0.328  0.820     0.454
      random rows    42      11251            41      0.240     0.290  0.816     0.460
grouped by client     1       2316            10      0.428     0.550  0.598     0.272
grouped by client     2       9390            10      0.308     0.382  0.722     0.366
grouped by client     3       6046            10      0.391     0.324  0.624     0.292
grouped by client     4      10432            10      0.195     0.380  0.486     0.362
grouped by client    42       8034            10      0.299     0.308  0.692     0.328
                   base_rate  baseline  mod

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from scipy.stats import spearmanr
R = pd.read_parquet("/content/ranking_lifecycle.parquet")[["content_hash_id", "impressions_30d"]]
D = D.merge(R, on="content_hash_id", how="left")

D["severity"] = -(D.impressions_last_30d / D.impressions_prev_30d - 1) * 100

# five honest features
f = D.impressions_90d - D.impressions_last_30d - D.impressions_prev_30d
D["impressions_first30"] = f
D["early_momentum_pct"] = (D.impressions_prev_30d / f.where(f >= 50) - 1) * 100
D["prev30_ctr_pct"] = D.clicks_prev_30d / D.impressions_prev_30d * 100
D["log_prev30_impressions"] = np.log1p(D.impressions_prev_30d)


# the leaky feature: outcome-window impressions, arriving through the join
D["LEAKY_log_joined_30d"] = np.log1p(D.impressions_30d)

honest = ["impressions_first30", "early_momentum_pct", "prev30_ctr_pct",
          "log_prev30_impressions", "age_at_outcome_start_days"]
leaky = honest + ["LEAKY_log_joined_30d"]

model = make_pipeline(SimpleImputer(strategy="median"),
                      QuantileTransformer(n_quantiles=200),
                      LinearRegression())

severe = (D.severity >= D.severity.quantile(.75)).to_numpy()
def p_at_k(s, k=50): return severe[np.argsort(-np.asarray(s))[:k]].mean()

for name, cols in [("honest (5)", honest), ("honest + LEAKY (6)", leaky)]:
    pred = cross_val_predict(model, D[cols], D.severity, groups=D.client_hash_id, cv=GroupKFold(5))
    print(name, spearmanr(pred, D.severity)[0], p_at_k(pred), p_at_k(pred, 500), severe.mean())


AttributeError: 'DataFrame' object has no attribute 'impressions_30d'

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
Observed: in these five splits, the random-row model reached precision@500 of 0.843 and the client-grouped model reached 0.630, on the same decline-only extract.
Measured: the model's lift over the base rate was +0.59 on random rows (0.843 vs 0.249) and +0.31 on grouped clients (0.630 vs 0.324), and rank correlation fell from 0.462 to 0.325.
Directional: a large share of the random-split advantage is associated with having other pages from the same client in training, but these runs don't isolate that cause.
Decision-support: plan around the grouped figure, which was 0.51 to 0.72 across splits, because a ranking built from the client-grouped model is the one to judge by when scoring clients it has never seen.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.